in this notebook we are going to re-implement the decoder only model but like the deepseek's type where we use Multihead Latent Attention, Rotary Position Encoding, Sparse Mixture of Experts, etc.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import random
import math
import wandb
import torch.nn.functional as F
from dataclasses import dataclass
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

In [ ]:
class configurations:
    model_dim = 256
    n_decoder_block = 16
    n_heads = 4
    head_dim = model_dim // n_heads
    dim_kv = 128
    dim_latent_q = model_dim//2
    dim_rope = 32
    total_Expert = 4
    top_k_expert = 3
    dropout_p = 0.1

    max_input_len = 256
    vocab_size = 12000
    train_batch_size = 28
    valid_batch_size = 8

    use_kv_cache = False

    epochs = 25
    lr = 5e-4
    device = "cuda" if torch.cuda.is_available() else "cpu"

cfg = configurations()

In [ ]:
def get_wandb_run():
  wandb.login(key=" ")
  run = wandb.init(
    entity=" ",
    project=" ",
  )
  return run

# **token embedding**

In [ ]:
class tokenembedding(nn.Module):

    "it maps the descrete token ids into the vector representations"

    def __init__(self, cfg):
        super().__init__()
        self.embeddings = nn.Embedding(cfg.vocab_size, cfg.model_dim)

    def forward(self, x):
        token_emb = self.embeddings(x)
        return token_emb

# **root mean sqare normalisation**

In [ ]:
class RMSNormalisation(nn.Module):

    "normalises the vector embeddings and rescaling so that the embeddings importance remain preserved."

    def __init__(self, cfg):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(cfg.model_dim))
        self.eps = 1e-8

    def forward(self, x):
        deno = torch.sqrt(torch.sum(torch.square(x),dim = -1, keepdim=True)/x.size(-1) + self.eps)
        x = x / deno
        x = self.gamma*x
        return x

# **projection layer: it projects the contextualized vector with next-probable token**

in the sentence "river bank is important for disaster" . if we pass the f(important) which is the contexualised form of all the sentence till there, now if we do f(important)*raw_embedding(for) it should be high

In [ ]:
class OutputProjection(nn.Module):

     def __init__(self, cfg):

        super().__init__()
        self.embeddings = nn.Linear(cfg.model_dim, cfg.vocab_size)

     def forward(self, x):
        x = self.embeddings(x)
        return x

# **rotary position embedding**

In [ ]:
class RotaryPositionEmbedding(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        self.cfg = cfg
        power = torch.arange(0, cfg.dim_rope, 2, device=cfg.device) / cfg.dim_rope
        inv_freq = 1.0/(10000**power)
        pos = torch.arange(0, cfg.max_input_len, device=cfg.device).unsqueeze(-1)
        angle = pos * inv_freq

        self.register_buffer("sin", torch.sin(angle))
        self.register_buffer("cos", torch.cos(angle))

    def forward(self, x, pos = None):        # x = [B, H, T, D]

        B, H, T, D = x.size()

        if not self.cfg.use_kv_cache:
          sinmat = self.sin[:T, :].unsqueeze(0).unsqueeze(0)               # type: ignore
          cosmat = self.cos[:T, :].unsqueeze(0).unsqueeze(0)               # type: ignore

        else:
          sinmat = self.sin[pos :pos + T, :].unsqueeze(0).unsqueeze(0)    # type: ignore
          cosmat = self.cos[pos :pos + T, :].unsqueeze(0).unsqueeze(0)    # type: ignore

          # print(f'pos: {pos} | T: {T} | x_size: {x.size()} | sinmat_size: {sinmat.size()}')

        # Reshape for RoPE application: [B, H, T, D/2, 2]
        x_reshaped = x.view(B, H, T, D // 2, 2)

        # Extract the two interleaving parts
        x1 = x_reshaped[..., 0]
        y1 = x_reshaped[..., 1]

        # Apply rotation
        x1_rotated = x1 * cosmat - y1 * sinmat
        y1_rotated = x1 * sinmat + y1 * cosmat

        # Stack them back together and reshape to the original [B, H, T, D] shape
        output = torch.stack([x1_rotated, y1_rotated], dim=-1)
        output = output.view(B, H, T, D)

        return output


it has Q.K = [Qc:Qr] * [Kc:Kr]
there Qc and Kc is the contextualise with no RoPE and Qr and Kr is with the RoPE.

in RoPE, we share the same weights of the downprojected while in No_Rope section each head has different learnable parameters unlike RoPE

In [ ]:
class MultiHeadLatentAttention(nn.Module):

    def __init__(self, cfg):

        super().__init__()
        self.cfg = cfg

        self.attn_dropout = nn.Dropout(0.1)
        self.RoPE = RotaryPositionEmbedding(cfg)

        # ======================= NoPE ===========================================================#

        self.W_dQ = nn.Linear(cfg.model_dim, cfg.dim_latent_q, bias = False)
        self.W_uQ = nn.Linear(cfg.dim_latent_q, cfg.model_dim, bias = False)

        self.W_dKV = nn.Linear(cfg.model_dim, cfg.dim_kv, bias = False)
        self.W_uK = nn.Linear(cfg.dim_kv, cfg.model_dim, bias = False)
        self.W_uV = nn.Linear(cfg.dim_kv, cfg.model_dim, bias = False)

        # ======================= RoPE ===========================================================#

        self.W_dQ_rope = nn.Linear(cfg.model_dim, cfg.dim_latent_q, bias = False)
        self.W_uQ_rope = nn.Linear(cfg.dim_latent_q, cfg.n_heads*cfg.dim_rope, bias = False)

        self.W_dK_rope = nn.Linear(cfg.model_dim, cfg.dim_rope, bias = False)

        # ======================= Output-Proj ====================================================#

        self.o_mat = nn.Linear(cfg.model_dim, cfg.model_dim, bias = False)

        # ======================= KV-Cache =======================================================#

        self.W_absorbed = None
        self.W_val_absorbed = None

        self.Cache_KV = None
        self.Cache_K_RoPE = None

    def reset_kv_cache(self):
        self.Cache_KV = None
        self.Cache_K_RoPE = None

    def forward(self,*, x, attention_mask):

      # if self.training and self.cfg.use_kv_cache:
        if not self.cfg.use_kv_cache:

              B, T, C = x.size()

              # ============ RoPE ============ #

              Q_C_RoPE    = self.W_dQ_rope(x)
              Q_pre_RoPE  = self.W_uQ_rope(Q_C_RoPE).view(B, T, self.cfg.n_heads, self.cfg.dim_rope).permute(0, 2, 1, 3)
              Q_RoPE = self.RoPE(Q_pre_RoPE)

              K_pre_RoPE = self.W_dK_rope(x)[:, None, :, :]
              k_RoPE = self.RoPE(K_pre_RoPE)           # [B, T, 256->32]

              attn_RoPE = Q_RoPE @ k_RoPE.transpose(-2, -1)

              # ============ NoPE ============ #

              Q_compressed_NoPE = self.W_dQ(x)
              Q_C = self.W_uQ(Q_compressed_NoPE).view(B, T, self.cfg.n_heads, self.cfg.head_dim).permute(0, 2, 1, 3)

              Cache_KV = self.W_dKV(x)
              K_C = self.W_uK(Cache_KV).view(B, T, self.cfg.n_heads, self.cfg.head_dim).permute(0, 2, 1, 3)
              V_C = self.W_uV(Cache_KV).view(B, T, self.cfg.n_heads, self.cfg.head_dim).permute(0, 2, 1, 3)

              attn_NoPE = Q_C @ K_C.transpose(-2, -1)

              scale = math.sqrt(self.cfg.head_dim + self.cfg.dim_rope)

              attn_score = (attn_RoPE + attn_NoPE)
              attn_score = attn_score / scale

              causual_mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=self.cfg.device))[None, None,:, :]             # [T, T]
              padding_mask = attention_mask[:, None, None, :].bool()                                  # [B, T]

              mask = causual_mask&padding_mask

              attn_score = attn_score.masked_fill(~mask, float('-inf'))

              norm_attn_score = attn_score.softmax(dim=-1)

              attn_dropped = self.attn_dropout(norm_attn_score)                                      # [B, H, T, T]

              output = attn_dropped@V_C

              output = output.permute(0, 2, 1, 3).reshape(B, T, -1)

        # =============================== INFERENCE ===============================

        else:

              B, T, C = x.size()

              if self.W_absorbed is None:
                  # self.W_absorbed = self.W_uQ.weight.transpose(-2, -1) @ self.W_uK.weight
                  WuQ_heads = self.W_uQ.weight.view(self.cfg.n_heads, self.cfg.head_dim, self.cfg.dim_latent_q)               #[4, 64, 128]
                  WuK_heads = self.W_uK.weight.view(self.cfg.n_heads, self.cfg.head_dim, self.cfg.dim_kv)                     #[4, 64, d_kv]
                  self.W_absorbed = torch.bmm(
                      WuQ_heads.transpose(-2, -1),                                                                #[4, 128, 64]@[4, 64, d_kv] = [4, 128, d_kv]
                      WuK_heads
                  )

                  #                                  [128, 256] --> [128, 4, 64] --> [4, 128, 64]
                  transpose_W_val_absorbed = self.W_uV.weight.transpose(-2, -1)
                  self.W_val_absorbed = transpose_W_val_absorbed.view(transpose_W_val_absorbed.size(0), self.cfg.n_heads, self.cfg.head_dim).permute(1, 0, 2)

              pos = 0 if self.Cache_KV is None else self.Cache_KV.size(-2)

              C_kv = self.W_dKV(x)

              # ================= RoPE ================= #

              C_q = self.W_dQ_rope(x)
              Q_pre_rope  = self.W_uQ_rope(C_q).view(B, T, self.cfg.n_heads, self.cfg.dim_rope).permute(0, 2, 1, 3)
              Q_rope = self.RoPE(Q_pre_rope, pos)

              K_pre_RoPE = self.W_dK_rope(x)[:, None, :, :]
              K_RoPE = self.RoPE(K_pre_RoPE, pos)

              if self.Cache_KV is None:
                  self.Cache_KV = C_kv
                  self.Cache_K_RoPE = K_RoPE

              else:
                  self.Cache_KV = torch.cat([self.Cache_KV, C_kv], dim = -2)
                  self.Cache_K_RoPE = torch.cat([self.Cache_K_RoPE, K_RoPE], dim = -2)

              C_KV = self.Cache_KV
              K_RoPE_latent = self.Cache_K_RoPE

              attn_rope =  Q_rope@K_RoPE_latent.transpose(-2, -1)

              # ================= NoPE ================= #

              Q_down = x@self.W_dQ.weight.transpose(-2, -1)                                                   # [ 1, 3, 256] @ [256, 128] == [1, 3, 128]

              Q_Compressed = Q_down.unsqueeze(1) @ self.W_absorbed                                            # [1, 3, 128]@[4, 128, 128] == [4, 3, 128]
              attn_ctxt = Q_Compressed@C_KV.unsqueeze(1).transpose(-2, -1)                                    # [4, 3, 128]@[1, 128, 10] == [4, 3, 10]

              scale = math.sqrt(self.cfg.head_dim + self.cfg.dim_rope)

              attn_score = (attn_rope + attn_ctxt) / scale

              T_q = Q_down.size(-2)
              T_k = C_KV.size(-2)

              if T_q == 1:
                causal_mask = torch.ones(T_q, T_k, dtype=torch.bool, device=x.device)[None, None,:, :]             # [T, T]
              else:
                causal_mask = torch.tril(torch.ones(T_q, T_k, dtype=torch.bool, device=x.device)[None, None,:, :])             # [T, T]

              attn_score = attn_score.masked_fill(~causal_mask, float('-inf'))
              attn_score = attn_score.softmax(dim = -1)                                                # [B, H, T, T]

              val = C_KV.unsqueeze(1)@self.W_val_absorbed.unsqueeze(0)

              output = attn_score@val

              output = output.permute(0, 2, 1, 3).reshape(x.size(0),T_q, -1)

        output = self.o_mat(output)

        return output

In [ ]:
class SwiGLUFFN(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.w1 = nn.Linear(cfg.model_dim, int(cfg.model_dim*(8/3)), bias = False)
        self.w2 = nn.Linear(cfg.model_dim, int(cfg.model_dim*(8/3)), bias = False)
        self.w3 = nn.Linear(int(cfg.model_dim*(8/3)), cfg.model_dim, bias = False)

    def forward(self, x):
        x = F.silu(self.w1(x)) * self.w2(x)
        x = self.w3(x)
        return x

class RouterClass(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.layer = nn.Linear(cfg.model_dim, cfg.total_Expert, bias=False)

    def forward(self, x):
        return self.layer(x)

In [ ]:
class MixtureofExperts(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.SharedExpert = SwiGLUFFN(cfg)
        self.Router = RouterClass(cfg)

        self.register_buffer("bias", torch.zeros(cfg.total_Expert, device=cfg.device))

        hidden = int(cfg.model_dim * (8 / 3))
        E = cfg.total_Expert
        C = cfg.model_dim

        self.w1 = nn.Parameter(torch.randn(E, hidden, C) * 0.02)
        self.w2 = nn.Parameter(torch.randn(E, hidden, C) * 0.02)
        self.w3 = nn.Parameter(torch.randn(E, C, hidden) * 0.02)

        self.update_bias_alpha = 0.001

    def forward(self, x):
        B, T, C = x.size()
        N = B * T
        E = self.cfg.total_Expert
        k = self.cfg.top_k_expert

        shared_expert = self.SharedExpert(x)

        x_flat = x.reshape(N, C)

        routes = self.Router(x_flat)                          # [N, E]
        scores = routes + self.bias                           # [N, E]
        expert_idx = scores.topk(k, dim=-1).indices          # [N, k]
        probs = routes.gather(-1, expert_idx).softmax(dim=-1) # [N, k]

        flat_expert = expert_idx.reshape(-1)                  # [N*k]
        flat_probs = probs.reshape(-1, 1)                     # [N*k, 1]
        token_ids = torch.arange(N, device=x.device).repeat_interleave(k)

        # Group routes by expert
        sorted_expert, perm = flat_expert.sort()
        token_ids_sorted = token_ids[perm]
        probs_sorted = flat_probs[perm]

        # repeat token activations only (OK); do NOT repeat weights
        x_rep = x_flat.repeat_interleave(k, dim=0)
        x_sorted = x_rep[perm]

        counts = torch.bincount(sorted_expert, minlength=E)

        out_sorted = torch.empty_like(x_sorted)
        start = 0
        for e, count in enumerate(counts.tolist()):
            if count == 0:
                continue

            end = start + count
            x_e = x_sorted[start:end]          # tokens routed to expert e
            p_e = probs_sorted[start:end]       # gate probs for those tokens

            h1 = F.silu(F.linear(x_e, self.w1[e])) * F.linear(x_e, self.w2[e])
            h2 = F.linear(h1, self.w3[e])

            out_sorted[start:end] = h2 * p_e
            start = end

        # scatter contributions back to original tokens
        output = torch.zeros_like(x_flat)
        output.index_add_(0, token_ids_sorted, out_sorted)
        output = output.view(B, T, C)

        final_output = shared_expert + output

        if self.training:
            self.update_bias(expert_idx)

        return final_output

    @torch.no_grad()
    def update_bias(self, expert_idx):
        expert_counts = torch.bincount(
            expert_idx.reshape(-1),
            minlength=self.cfg.total_Expert
        ).float()

        self.bias.add_(self.update_bias_alpha * (expert_counts.mean() - expert_counts))

In [ ]:
class DecoderBlock(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        self.cfg = cfg
        self.RMSNorm1 = RMSNormalisation(cfg=cfg)
        self.RMSNorm2 = RMSNormalisation(cfg=cfg)
        self.MHLA = MultiHeadLatentAttention(cfg=cfg)
        self.MoEs = MixtureofExperts(cfg=cfg)

    def forward(self, x, attention_mask):

        x = x + self.MHLA(x = self.RMSNorm1(x), attention_mask=attention_mask)
        x = x + self.MoEs(self.RMSNorm2(x))

        return x

In [ ]:
class Model(nn.Module):

    def __init__(self, cfg):

        super().__init__()
        self.cfg = cfg
        self.tknEmbedding = tokenembedding(cfg)
        self.DecoderBlocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_decoder_block)])
        self.endRMSNorms = RMSNormalisation(cfg=cfg)

        self.output_projection = nn.Linear(cfg.model_dim, cfg.vocab_size)
        self.output_projection.weight = self.tknEmbedding.embeddings.weight

    def forward(self,*, input_ids, attention_mask=None):

        x = self.tknEmbedding(input_ids)

        for layer in self.DecoderBlocks:
            x = layer(x, attention_mask)

        x = self.endRMSNorms(x)
        logits = self.output_projection(x)

        return logits

    def generate(self, input_ids, max_new_token = 10,  tempreture = 1.0):

        self.eval()
        for layer in self.DecoderBlocks:
            layer.MHLA.reset_kv_cache()

        generate_tok_len = max_new_token - input_ids.size(-1)

        output_token_ids = input_ids.clone()

        for _ in range(generate_tok_len):
            logits = self(input_ids = input_ids, attention_mask=None)[:, -1, :] / tempreture
            norm_logits = F.softmax(logits, dim = -1)
            tok_index = norm_logits.multinomial(1)
            output_token_ids = torch.cat([output_token_ids, tok_index], dim = -1)
            input_ids = tok_index

        return output_token_ids

In [ ]:
deepseek_lite = Model(cfg).to(cfg.device)

In [ ]:
def init_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

    elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

_ = deepseek_lite.apply(init_weights)

In [ ]:
sum(p.numel() for p in deepseek_lite.parameters())/1e6

# **Dataset**

In [ ]:
from datasets import load_dataset

ds = load_dataset("arsalanaa/children_story_dataset")

In [ ]:
idx = torch.randperm(len(ds['train']))
train_idx = idx[:20000]
valid_idx = idx[20000: 25000]

In [ ]:
train_ds = ds['train']['text'][train_idx]
valid_ds = ds['train']['text'][valid_idx]

In [ ]:
from tokenizers import ByteLevelBPETokenizer
import itertools

# Display a few examples from the training dataset
print("--- First 5 examples from train_ds ---")
for i in range(5):
    print(f"Example {i+1}: {train_ds[i][:200]}...") # Print first 200 chars to avoid very long outputs
print("------------------------------------")

# Combine the training and validation datasets for tokenizer training
all_texts = itertools.chain(train_ds, valid_ds)

# Initialize a ByteLevelBPETokenizer
# We'll use the vocab_size defined in your configurations (cfg.vocab_size)
# Special tokens are kept consistent with the GPT2 tokenizer you're using.
tokenizer_trainer = ByteLevelBPETokenizer(
    add_prefix_space=True, # Important for ByteLevelBPE to handle leading spaces
    lowercase=False # Match GPT2 default behavior
)

# Define special tokens. GPT2 uses <|endoftext|> for various roles.
# We'll use <unk> for unknown tokens and <pad> for padding, if not already covered.
# Given the current tokenizer has <|endoftext|> as eos, bos, unk, pad, we'll mimic that.
special_tokens = ["<|endoftext|>"]

# Train the tokenizer
# `train_from_iterator` expects an iterator that yields strings
tokenizer_trainer.train_from_iterator(
    all_texts,
    vocab_size=cfg.vocab_size, # Using vocab_size from your configurations
    min_frequency=2,
    special_tokens=special_tokens
)

# Save the tokenizer directly as tokenizer.json
output_dir = "."
tokenizer_trainer.save("tokenizer.json")

print(f"\nBPE tokenizer training complete! Saved as tokenizer.json in {output_dir}/")
print("You can load this tokenizer using: `PreTrainedTokenizerFast(tokenizer_file='tokenizer.json')`")

In [ ]:
from transformers import PreTrainedTokenizerFast

# Load the trained tokenizer from the tokenizer.json file
# This assumes 'tokenizer.json' is present in the current working directory
tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="tokenizer.json",
    # Configure special tokens to match GPT2 defaults and your training setup
    eos_token="<|endoftext|>",
    bos_token="<|endoftext|>",
    unk_token="<|endoftext|>",
    pad_token="<|endoftext|>"
)

# The pad_token and pad_token_id are already set if provided in the constructor above.
# These lines ensure explicit setting for clarity/consistency if needed, though they might be redundant.
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Tokenizer loaded successfully: {tokenizer.__class__.__name__}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Special tokens map: {tokenizer.special_tokens_map}")

In [ ]:
class Datasets(Dataset):
    def __init__(self, cfg, dataset, tokenizer):
        self.cfg = cfg
        self.texts = dataset
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokenized = self.tokenizer(self.texts[idx],  max_length=self.cfg.max_input_len+1, truncation=True, padding="max_length", return_tensors="pt")

        input_ids = tokenized['input_ids'].squeeze(0)
        attention_mask = tokenized['attention_mask'].squeeze(0)

        x = input_ids[:-1]
        y = input_ids[1:].clone()
        y[y==self.tokenizer.pad_token_id] = -100
        return {
            "input_ids": x,
            "target_ids": y,
            "attention_mask": attention_mask[:-1]
        }

In [ ]:
TrainDataset = Datasets(cfg, train_ds, tokenizer)
ValidDataset = Datasets(cfg, valid_ds, tokenizer)

In [ ]:
my_train_dataloader = DataLoader(TrainDataset, batch_size=cfg.train_batch_size, shuffle=True)
my_valid_dataloader = DataLoader(ValidDataset, batch_size=cfg.valid_batch_size, shuffle=False)

In [ ]:
for batch in my_train_dataloader:
  i, y, b = batch['input_ids'], batch['target_ids'], batch['attention_mask']
  break

In [ ]:
b.size()

# **Training Loops**

In [ ]:
runs = get_wandb_run()

In [ ]:
weights_decay = []
no_weights_decay = []

for names, params in deepseek_lite.named_parameters():
     if params.ndim ==1 or names.endswith("bias") or names.endswith("gamma") or names.endswith("embeddings.weight"):
        no_weights_decay.append(params)
     else:
        weights_decay.append(params)

In [ ]:
def lr_lambda(step):
    total_steps = len(my_train_dataloader)*cfg.epochs
    warmup_steps = int(0.1* total_steps)
    minimum_lr = 5e-5

    if step < warmup_steps:
        return step / warmup_steps
    return minimum_lr + 0.5 * (1 + math.cos(math.pi * (step - warmup_steps) / (total_steps - warmup_steps)))

In [ ]:
criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)
optimizers = torch.optim.AdamW([{"params": weights_decay, "weight_decay": 0.1},
                                {"params": no_weights_decay, "weight_decay": 0.0}],
                               lr = cfg.lr, betas=(0.9, 0.95), eps=1e-8)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizers, lr_lambda)

In [ ]:
for layer in deepseek_lite.DecoderBlocks:
    layer.MHLA.reset_kv_cache()

In [ ]:
cfg.train_batch_size = 32 # Reduced batch size to fit in memory
cfg.valid_batch_size = 8  # Keep valid_batch_size consistent or reduce as well

# Re-instantiate dataloaders with the new batch sizes
my_train_dataloader = DataLoader(TrainDataset, batch_size=cfg.train_batch_size, shuffle=True)
my_valid_dataloader = DataLoader(ValidDataset, batch_size=cfg.valid_batch_size, shuffle=False)

for epoch in range(cfg.epochs):

    deepseek_lite.train()
    for train_batch in my_train_dataloader:


        input_ids = train_batch['input_ids'].to(cfg.device)
        target_ids = train_batch['target_ids'].to(cfg.device)
        attention_mask = train_batch['attention_mask'].to(cfg.device)

        logits = deepseek_lite(input_ids=input_ids, attention_mask=attention_mask)

        loss = criterion(logits.view(-1, logits.size(-1)), target_ids.reshape(-1))

        optimizers.zero_grad()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(deepseek_lite.parameters(), 1.0)
        optimizers.step()
        scheduler.step()

        runs.log(
            {
                "training_loss": loss.item(),
                "learning-rate": optimizers.param_groups[0]['lr']
            }
        )

    print(f" === Set to Evaluations! after epochs: {epoch+1} === ")
    deepseek_lite.eval()
    with torch.no_grad():
         eval_loss = 0
         eval_steps = 0
         for valid_batch in my_valid_dataloader:

            input_ids = valid_batch['input_ids'].to(cfg.device)
            target_ids = valid_batch['target_ids'].to(cfg.device)
            attention_mask = valid_batch['attention_mask'].to(cfg.device)

            logits = deepseek_lite(input_ids=input_ids, attention_mask=attention_mask)

            running_loss = criterion(logits.view(-1, logits.size(-1)), target_ids.reshape(-1))

            eval_loss += running_loss.item()
            eval_steps += 1

            for layer in deepseek_lite.DecoderBlocks:
                layer.MHLA.reset_kv_cache()

         runs.log({
             "eval_loss": eval_loss/eval_steps
         })

In [ ]:
torch.save(deepseek_lite.state_dict(), "deepseek_lite.pt")

In [ ]:
deepseek_lite.load_state_dict(torch.load("deepseek_lite.pt"))

In [ ]:
cfg.use_kv_cache = True

In [73]:
def generate(text, max_new_token = 100,  tempreture = 0.3):


    input_ids = tokenizer(f"{text}", return_tensors='pt')['input_ids'].to("cuda")
    msk = tokenizer(f"{text}", return_tensors='pt')['attention_mask'].to("cuda")

    deepseek_lite.eval()
    for layer in deepseek_lite.DecoderBlocks:
        layer.MHLA.reset_kv_cache()

    generate_tok_len = max_new_token - input_ids.size(-1)

    output_token_ids = input_ids.clone()

    for _ in range(generate_tok_len):
        logits = deepseek_lite(input_ids = input_ids, attention_mask=None)[:, -1, :] / tempreture
        norm_logits = F.softmax(logits, dim = -1)
        tok_index = norm_logits.multinomial(1)
        output_token_ids = torch.cat([output_token_ids, tok_index], dim = -1)
        input_ids = tok_index

    return output_token_ids

In [84]:
valid_ds[555]

' Molly and Sam were two best friends who loved watching movies together. One day, they watched a film called "The Startling Story." After the movie ended, Molly turned to Sam and asked, "What did you think of the movie?"\n\nSam thought for a moment before answering, "I think the word \'startling\' describes it perfectly! It was full of surprises and twists we didn\'t see coming."\n\nMolly nodded her head in agreement. "Yes, it definitely kept us guessing!" she said. "But do you know what the word \'startling\' means?"\n\nSam shook his head no. Molly explained, "It means something that causes sudden surprise or shock. And when we use it in the context of a movie, it usually has a positive meaning. That\'s because people like being surprised by stories, especially ones that keep us engaged and interested."\n\nAs they continued talking, they realized that many things around them could also be described as startling - such as lightning during a storm, or a rabbit suddenly jumping out of b

In [75]:
_ = deepseek_lite.to(cfg.device)

In [86]:
o_ids = generate(" Molly and Sam were two best friends")[0]

In [87]:
o_ids

tensor([3308,  285,  708,  504,  652,  699,  495,  590,  624,  284,  716,  605,
          14,  669,  432,   12,  304,  646,  284, 1761,  688,  563,  490, 7330,
         260,  286,  768,  688,   12,  436,  304,  643,  703, 1134, 1635,  263,
        1709,  309, 3670,  304,  506, 1341, 8531,  391, 2438,    9,  285,  263,
         563,  391, 1756,  880,   14,  199,  199, 8959,  506,  357, 1433,   12,
         333, 1722,   12, 1106,  403, 1122,  469,  993, 3809,    1, 1304,  539,
        1056,  469, 2532, 1734,  284,  726,  502,  263, 2777, 1709,  309, 3670,
         396, 1038,  459,  888,  304,  670,   14,  708,  838, 1052,   12, 1691,
         284, 2858,  502,  554], device='cuda:0')

In [88]:
print(tokenizer.decode(o_ids))

 Molly and Sam were two best friends who loved to play together. One day, they decided to challenge each other by picking a penny each, so they would need three times the number of steps they had (which was 5) and the other was going down.

Molly had an idea, "Sam, let's set our own goal! We can use our math skills to find out the total number of steps we have." So they did. Sam thought hard, trying to figure out what


In [ ]:
o_ids